# Tarefa 4 - Ajuste do preditor de degradacao

Este notebook ajusta uma floresta aleatoria para prever a classe do fluxo **12 minutos a frente**. A classe atual serve como referencia de persistencia, nao como alvo do modelo.

In [14]:
import pandas as pd
import numpy as np
from pathlib import Path
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import export_text
from sklearn.metrics import confusion_matrix, f1_score, precision_recall_fscore_support

_v = tuple(int(x) for x in sklearn.__version__.split(".")[:2])
assert _v >= (1, 4), "Atualize o scikit-learn para >= 1.4: o Random Forest precisa aceitar NaN nas features."

# Localizando os dados de treino e validacao
CURRENT_DIR = Path.cwd().resolve()
ARQUIVOS_ENTRADA = ("treino.csv", "validacao.csv")
ROOT = next(
    (
        candidate
        for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
        if all(
            (candidate / "data_ripe_atlas" / "processed" / filename).is_file()
            for filename in ARQUIVOS_ENTRADA
        )
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Could not find treino.csv and validacao.csv in data_ripe_atlas/processed "
        "from the current directory."
    )
PASTA_SPLITS = ROOT / "data_ripe_atlas" / "processed"

CAMINHO_TREINO = PASTA_SPLITS / "treino.csv"
CAMINHO_VALIDACAO = PASTA_SPLITS / "validacao.csv"

tr = pd.read_csv(CAMINHO_TREINO)
va = pd.read_csv(CAMINHO_VALIDACAO)

Preparacao do alvo futuro e das features temporais. O mesh mede aproximadamente a cada 240 segundos; portanto, o alvo fica na terceira medicao seguinte do mesmo fluxo (aproximadamente 12 minutos). Para nao pular lacunas grandes, so usamos pares cuja distancia observada esteja entre 11 e 13 minutos. As features usam apenas a medicao atual e o historico anterior.

In [15]:
HORIZONTE_SEGUNDOS = 12 * 60
PASSOS_FUTUROS = 3
TOLERANCIA_HORIZONTE_SEGUNDOS = 60

# Preserve o indice original para separar treino e validacao depois do lag causal.
tr = tr.copy().reset_index(drop=True)
va = va.copy().reset_index(drop=True)
tr["timestamp"] = pd.to_numeric(tr["timestamp"], errors="raise")
va["timestamp"] = pd.to_numeric(va["timestamp"], errors="raise")
tr["particao"] = "treino"
va["particao"] = "validacao"

# O lag da primeira linha da validacao pode usar a observacao anterior do treino.
historico = pd.concat([tr, va], ignore_index=True).sort_values(
    ["fluxo_id", "timestamp"]
).copy()
grupos_historico = historico.groupby("fluxo_id", sort=False)
historico["z_robusto_t_menos_1"] = grupos_historico["z_robusto"].shift(1)
historico["delta_z_robusto"] = historico["z_robusto"] - historico["z_robusto_t_menos_1"]

tr_lag = historico[historico["particao"] == "treino"].drop(columns="particao").copy()
va_lag = historico[historico["particao"] == "validacao"].drop(columns="particao").copy()


def adicionar_alvo_futuro(df):
    df = df.sort_values(["fluxo_id", "timestamp"]).copy()
    grupos = df.groupby("fluxo_id", sort=False)
    df["status_futuro"] = grupos["classe"].shift(-PASSOS_FUTUROS)
    df["timestamp_futuro"] = grupos["timestamp"].shift(-PASSOS_FUTUROS)
    df["horizonte_observado_segundos"] = df["timestamp_futuro"] - df["timestamp"]
    dentro_do_horizonte = df["horizonte_observado_segundos"].between(
        HORIZONTE_SEGUNDOS - TOLERANCIA_HORIZONTE_SEGUNDOS,
        HORIZONTE_SEGUNDOS + TOLERANCIA_HORIZONTE_SEGUNDOS,
    )
    return df.loc[dentro_do_horizonte & df["status_futuro"].notna()].copy()


tr_pred = adicionar_alvo_futuro(tr_lag)
va_pred = adicionar_alvo_futuro(va_lag)
if tr_pred.empty or va_pred.empty:
    raise ValueError("Nao ha pares de medicacoes suficientes para construir o alvo futuro de 12 minutos.")

print("Horizonte-alvo: 12 minutos (terceira medicao seguinte, tolerancia de +/- 60 s)")
print("Linhas elegiveis - treino:", len(tr_pred), "de", len(tr_lag))
print("Linhas elegiveis - validacao:", len(va_pred), "de", len(va_lag))
print("Horizonte observado na validacao (segundos):")
print(va_pred["horizonte_observado_segundos"].describe().round(1).to_string())

Horizonte-alvo: 12 minutos (terceira medicao seguinte, tolerancia de +/- 60 s)
Linhas elegiveis - treino: 95166 de 95406
Linhas elegiveis - validacao: 37646 de 38138
Horizonte observado na validacao (segundos):
count    37646.0
mean       720.0
std          6.1
min        703.0
25%        716.0
50%        720.0
75%        724.0
max        737.0


Definicao das features e do alvo. O modelo recebe as metricas da medicao atual e uma variacao calculada apenas com a medicao anterior. O alvo e status_futuro, a classe da terceira medicao seguinte; classe atual fica fora das features e sera usada somente como comparacao de persistencia.

In [16]:
NOVAS_FEATURES = [
    "z_robusto_t_menos_1",
    "delta_z_robusto",
    "z_robusto",
    "perda_pct",
    "jitter_relativo",
    "n5_risco",
    "n5_timeout",
    "n5_aumento80",
]
ALVO = "status_futuro"
STATUS_ATUAL = "classe"
CLASSES = ["OK", "RISCO", "FALHA"]

X_tr, y_tr = tr_pred[NOVAS_FEATURES], tr_pred[ALVO]
X_va, y_va = va_pred[NOVAS_FEATURES], va_pred[ALVO]
y_persistencia = va_pred[STATUS_ATUAL]

print("Horizonte previsto: 12 minutos")
print("Colunas de entrada:", NOVAS_FEATURES)
print("Alvo futuro:", ALVO)
print("Status atual usado apenas como baseline:", STATUS_ATUAL)

Horizonte previsto: 12 minutos
Colunas de entrada: ['z_robusto_t_menos_1', 'delta_z_robusto', 'z_robusto', 'perda_pct', 'jitter_relativo', 'n5_risco', 'n5_timeout', 'n5_aumento80']
Alvo futuro: status_futuro
Status atual usado apenas como baseline: classe


A floresta aleatoria estima `status_futuro` cerca de 12 minutos a frente. Uma grade pequena e comparada no treino e os hiperparametros sao selecionados pela macro-F1 da validacao. Tambem avaliamos `class_weight="balanced_subsample"`: essa opcao pode aumentar a deteccao de RISCO, mas reduzir a macro-F1. Como a validacao participa da selecao, estes resultados servem para ajuste; o teste permanece fechado para avaliacao final posterior. O baseline repete `classe` atual como previsao futura.

In [17]:
# Grade pequena; cada configuracao e ajustada somente no treino.
N_ESTIMATORS = 200
SEMENTE = 42
GRADE_HIPERPARAMETROS = [
    {
        "max_depth": profundidade,
        "min_samples_leaf": min_folha,
        "class_weight": peso_classe,
        "criterion": criterio,
    }
    for profundidade in [6, 10]
    for min_folha in [10, 50]
    for peso_classe in [None, "balanced_subsample"]
    for criterio in ["gini", "entropy"]
]

resultados_busca = []
melhor_chave = (-1.0, -1.0)
melhor_modelo = None
parametros_escolhidos = None
for parametros in GRADE_HIPERPARAMETROS:
    candidato = RandomForestClassifier(
        n_estimators=N_ESTIMATORS,
        random_state=SEMENTE,
        n_jobs=-1,
        **parametros,
    )
    candidato.fit(X_tr, y_tr)
    pred_validacao = candidato.predict(X_va)
    f1_validacao = f1_score(
        y_va, pred_validacao, labels=CLASSES, average="macro", zero_division=0
    )
    _, recall_validacao, _, _ = precision_recall_fscore_support(
        y_va, pred_validacao, labels=CLASSES, zero_division=0
    )
    acuracia_balanceada = sklearn.metrics.balanced_accuracy_score(y_va, pred_validacao)
    resultados_busca.append({
        "max_depth": parametros["max_depth"],
        "min_samples_leaf": parametros["min_samples_leaf"],
        "class_weight": parametros["class_weight"] or "None",
        "criterion": parametros["criterion"],
        "f1_macro": f1_validacao,
        "balanced_accuracy": acuracia_balanceada,
        "recall_risco": recall_validacao[CLASSES.index("RISCO")],
    })
    chave = (f1_validacao, acuracia_balanceada)
    if chave > melhor_chave:
        melhor_chave = chave
        melhor_modelo = candidato
        parametros_escolhidos = parametros.copy()

if melhor_modelo is None or parametros_escolhidos is None:
    raise RuntimeError("A busca de hiperparametros nao produziu um modelo valido.")

floresta_ajustada = melhor_modelo
resultados_busca = pd.DataFrame(resultados_busca).sort_values(
    ["f1_macro", "balanced_accuracy"], ascending=False
).reset_index(drop=True)
print("Resultados da busca na validacao (ordenados por F1 macro):")
print(resultados_busca.round(4).to_string(index=False))
print("\nParametros escolhidos:", parametros_escolhidos)
print("Criterio de selecao: maior F1 macro; desempate por balanced accuracy.")

profundidades = [arvore.get_depth() for arvore in floresta_ajustada.estimators_]
folhas = [arvore.get_n_leaves() for arvore in floresta_ajustada.estimators_]

print("Parametros finais da floresta aleatoria:")
print("Criterio:", floresta_ajustada.criterion)
print("Numero de arvores:", len(floresta_ajustada.estimators_))
print("Profundidade media das arvores componentes:", round(float(np.mean(profundidades)), 2))
print("Folhas medias por arvore componente:", round(float(np.mean(folhas)), 2))

Resultados da busca na validacao (ordenados por F1 macro):
 max_depth  min_samples_leaf       class_weight criterion  f1_macro  balanced_accuracy  recall_risco
        10                10               None      gini    0.7164             0.6686        0.3056
        10                10               None   entropy    0.7146             0.6666        0.3003
        10                50               None      gini    0.7098             0.6630        0.2856
        10                50               None   entropy    0.7085             0.6615        0.2803
         6                10               None      gini    0.7073             0.6609        0.2785
         6                50               None      gini    0.7053             0.6593        0.2725
         6                10               None   entropy    0.7005             0.6553        0.2578
         6                50               None   entropy    0.6878             0.6452        0.2253
        10                10 bal

Avalia-se na validacao a previsao de `status_futuro`, calculado com a terceira medicao seguinte (cerca de 12 minutos), contra o baseline de persistencia (repetir `classe` atual). A selecao prioriza macro-F1; balanced accuracy e recall de RISCO tambem devem ser considerados para expor o custo de erros na classe minoritaria. A validacao foi usada para escolher hiperparametros, portanto nao substitui a avaliacao final no teste.

In [18]:
pred_va = floresta_ajustada.predict(X_va)

mat = pd.DataFrame(
    confusion_matrix(y_va, pred_va, labels=CLASSES),
    index=[f"real {c}" for c in CLASSES],
    columns=[f"prevista {c}" for c in CLASSES],
)
mat_persistencia = pd.DataFrame(
    confusion_matrix(y_va, y_persistencia, labels=CLASSES),
    index=[f"real {c}" for c in CLASSES],
    columns=[f"prevista {c}" for c in CLASSES],
)

p, r, f, s = precision_recall_fscore_support(
    y_va, pred_va, labels=CLASSES, zero_division=0
)
metricas = pd.DataFrame(
    {"precisao": p, "recall": r, "F1": f, "medicoes": s}, index=CLASSES
)
f1_macro = f1_score(y_va, pred_va, labels=CLASSES, average="macro", zero_division=0)
f1_macro_persistencia = f1_score(
    y_va, y_persistencia, labels=CLASSES, average="macro", zero_division=0
)
balanced_accuracy = sklearn.metrics.balanced_accuracy_score(y_va, pred_va)
balanced_accuracy_persistencia = sklearn.metrics.balanced_accuracy_score(
    y_va, y_persistencia
)

print("--- PREVISAO DA CLASSE 12 MINUTOS A FRENTE ---")
print("Matriz de confusao - Random Forest:")
print(mat.to_string())
print("\nMetricas por classe - Random Forest:")
print(metricas.round(4).to_string())
print("\nMatriz de confusao - persistencia de status_atual:")
print(mat_persistencia.to_string())
print("\nComparacao no mesmo conjunto de validacao:")
print(f"Random Forest: F1 macro = {f1_macro:.4f} | balanced accuracy = {balanced_accuracy:.4f}")
print(f"Persistencia:  F1 macro = {f1_macro_persistencia:.4f} | balanced accuracy = {balanced_accuracy_persistencia:.4f}")
print(f"Diferenca de F1 macro (floresta - persistencia): {f1_macro - f1_macro_persistencia:+.4f}")

amostras_predicao = pd.DataFrame({
    "instante_atual_utc": pd.to_datetime(va_pred["timestamp"].to_numpy(), unit="s", utc=True),
    "instante_alvo_utc": pd.to_datetime(va_pred["timestamp_futuro"].to_numpy(), unit="s", utc=True),
    "status_atual": va_pred[STATUS_ATUAL].to_numpy(),
    "status_futuro_real": y_va.to_numpy(),
    "status_futuro_previsto": pred_va,
    "baseline_persistencia": y_persistencia.to_numpy(),
})
print("\nExemplos de previsao para 12 minutos a frente:")
print(amostras_predicao.head(10).to_string(index=False))

arvore_exemplo = export_text(
    floresta_ajustada.estimators_[0],
    feature_names=NOVAS_FEATURES,
    decimals=2,
    max_depth=3,
)
for indice, classe in enumerate(floresta_ajustada.classes_):
    arvore_exemplo = arvore_exemplo.replace(
        f"class: {float(indice)}", f"class: {classe}"
    )
print("\n--- EXEMPLO ILUSTRATIVO: ARVORE COMPONENTE 1 ---")
print(arvore_exemplo)

--- PREVISAO DA CLASSE 12 MINUTOS A FRENTE ---
Matriz de confusao - Random Forest:
            prevista OK  prevista RISCO  prevista FALHA
real OK           26327             216             294
real RISCO         2118            1023             206
real FALHA         1833             263            5366

Metricas por classe - Random Forest:
       precisao  recall      F1  medicoes
OK       0.8695  0.9810  0.9219     26837
RISCO    0.6811  0.3056  0.4219      3347
FALHA    0.9148  0.7191  0.8052      7462

Matriz de confusao - persistencia de status_atual:
            prevista OK  prevista RISCO  prevista FALHA
real OK           23873            1498            1466
real RISCO         1455            1426             466
real FALHA         1488             416            5558

Comparacao no mesmo conjunto de validacao:
Random Forest: F1 macro = 0.7164 | balanced accuracy = 0.6686
Persistencia:  F1 macro = 0.6866 | balanced accuracy = 0.6868
Diferenca de F1 macro (floresta - persisten